In [0]:
%run ../../config/variables

In [0]:
import sys
sys.path.append("..")
sys.path.append("../..")

from lib.job_manager import load_config, split_config
from lib_etl.s3 import etl_input_data_validator
import lib_etl.validations_ETL as validations

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)

## Data availability check

In [0]:
source_path = data_paths["intermediate"]["AH5"]
recency_lookback_duration = data_paths.get("recency_lookback_duration", {})
etl_input_data_validator(
    "intermediate",
    recency_lookback_duration,
    data_paths,
    [
        "AH5",
    ],
    spark
)

## Load source data

In [0]:
df = spark.read.parquet(source_path)
df.createOrReplaceTempView('df')

## Save to delta table

In [0]:
spark.sql(f"""
    INSERT OVERWRITE {bronze_AH5}
    SELECT
        AH5_CD,
        AH5_DESC,
        QTY_MBRS,
        TRIPS,
        UNITS,
        SALES,
        UNIT_RETAIL_PRICE,
        UNITS_PER_TRIP,
        UNITS_PER_MBR,
        TRIPS_PER_MBR,
        PENETRATION_RATE,
        QTY_MBRS_1,
        QTY_MBRS_2,
        QTY_MBRS_3,
        QTY_MBRS_4,
        QTY_MBRS_5,
        QTY_MBRS_6,
        QTY_MBRS_7,
        QTY_MBRS_8,
        QTY_MBRS_9,
        QTY_MBRS_10,
        QTY_MBRS_11,
        QTY_MBRS_12,
        MONTH_1_SEASONALITY,
        MONTH_2_SEASONALITY,
        MONTH_3_SEASONALITY,
        MONTH_4_SEASONALITY,
        MONTH_5_SEASONALITY,
        MONTH_6_SEASONALITY,
        MONTH_7_SEASONALITY,
        MONTH_8_SEASONALITY,
        MONTH_9_SEASONALITY,
        MONTH_10_SEASONALITY,
        MONTH_11_SEASONALITY,
        MONTH_12_SEASONALITY,
        PURCHASE_CYCLE_DAYS,
        PURCHASE_CYCLE,
        SCALED_MONTH_1_SEASONALITY,
        SCALED_MONTH_2_SEASONALITY,
        SCALED_MONTH_3_SEASONALITY,
        SCALED_MONTH_4_SEASONALITY,
        SCALED_MONTH_5_SEASONALITY,
        SCALED_MONTH_6_SEASONALITY,
        SCALED_MONTH_7_SEASONALITY,
        SCALED_MONTH_8_SEASONALITY,
        SCALED_MONTH_9_SEASONALITY,
        SCALED_MONTH_10_SEASONALITY,
        SCALED_MONTH_11_SEASONALITY,
        SCALED_MONTH_12_SEASONALITY,
        SCALED_PURCHASE_CYCLE,
        INCLUDE_OR_EXCLUDE,
        EXCLUSION_TYPE,
        EXCLUSION_SUBTYPE,
        SEASON_MONTH_1,
        SEASON_MONTH_2,
        SEASON_MONTH_3,
        SEASON_MONTH_4,
        SEASON_MONTH_5,
        SEASON_MONTH_6,
        SEASON_MONTH_7,
        SEASON_MONTH_8,
        SEASON_MONTH_9,
        SEASON_MONTH_10,
        SEASON_MONTH_11,
        SEASON_MONTH_12,
        PCT_GROSS_MARGIN,
        GROSS_MARGIN
    FROM df
""")